In [1]:
from gu_toolkit import *
from helpers.Fourier_Sounds_helper import *

In [2]:
fig_interaction = Figure(x_range=(0, 1), y_range=(-3, 3))
fig_interaction.show()
with fig_interaction:
    set_title(r"Products of trig functions")
    plot(sin(2*pi*u*x), x, label=r"$\sin(2 \pi u x)$",id="sin_u",alpha=0.3)
    plot(sin(2*pi*v*x), x, label=r"$\sin(2 \pi v x)$",id="sin_v", alpha=0.3)
    plot(sin(2*pi*u*x)*sin(2*pi*v*x), x, label="product", id="product",  width=5)
    parameter(u, value=0, min=0, max=20, step=1)
    parameter(v, value=0, min=0, max=20, step=1)

OneShotOutput()

## What we discovered

We have shown the following orthogonality relations on the interval $[0,1]$:

$$
\int_0^1 \sin(2\pi m x)\sin(2\pi n x)\,dx=
\begin{cases}
0,& m\ne n,\\[4pt]
\tfrac12,& m=n\ge 1,
\end{cases}
$$
$$
\int_0^1 \cos(2\pi m x)\cos(2\pi n x)\,dx=
\begin{cases}
0,& m\ne n,\\[4pt]
\tfrac12,& m=n\ge 1,
\end{cases}
$$
$$
\int_0^1 \sin(2\pi m x)\cos(2\pi n x)\,dx=0.
$$

## How do we use this?


**IF** we have 

$$
F(x)=a_0+\sum_{n=1}^{\infty} a_n \cos(2\pi n x)+b_n \sin(2\pi n x),
$$

then multiply by $\cos(2\pi m x)$ and integrate:

$$  
\int_0^1 F(x)\cos(2\pi m x)\,dx
\begin{aligned}[t]
&=
\bbox[#EEC0AB]{\int_0^1 a_0 \cos(2\pi  x)\cos(2\pi m x)\,dx}
\\
& \bbox[#EEC0AB]{+\int_0^1 a_1 \cos(2\pi  x)\cos(2\pi m x) \, dx}
\\
& \bbox[#EEC0AB]{+\int_0^1 a_2 \cos(2\pi \cdot 2 x)\cos(2\pi m x) \, dx}
\\ & \ldots
\\
& \bbox[#B2D58C]{+\int_0^1 a_m \cos(2\pi \cdot m x)\cos(2\pi m x) \, dx}
\\
& \ldots
\end{aligned}
\quad
\begin{aligned}[t]
&+
\bbox[#EEC0AB]{\int_0^1 b_1 \sin(2\pi n x)\cos(2\pi m x)\,dx}
\\
& \bbox[#EEC0AB]{+\int_0^1 a_1 \cos(2\pi  x)\cos(2\pi m x) \, dx}
\\
&\bbox[#EEC0AB]{ +\int_0^1 a_2 \cos(2\pi \cdot 2 x)\cos(2\pi m x) \, dx}
\\ & \ldots
\\
& \bbox[#EEC0AB]{+\int_0^1 a_m \cos(2\pi \cdot m x)\cos(2\pi m x) \, dx}
\\ &
+\ldots
\end{aligned}
$$



So we found


$$
a_m=2\int_0^1 F(x)\cos(2\pi m x)\,dx,
$$
---

and similarly


$$
b_m=2\int_0^1 F(x)\sin(2\pi m x)\,dx,
$$
---


## Lets compute!

### The function $F$

In [3]:
# Compute Fourier coefficients of the mystery function using explicit numerical integration.
Nmax = 5
F=create_mystery_function(Nmax)
an_F, bn_F = fourier_coefficients(F.expr, Nmax, interval=(0, 1))

## Display coefficients
display(Markdown("Fourier coefficients for the mystery function $F$:"))

title_line=[
    f"& \\cos\\quad \\text{{coefficients}}&&\\sin \\quad\\text{{coefficients}}"
]
lines = [
    f"&a_{{{n}}} = {an_F[n]:.6f}  & &b_{{{n}}} = {bn_F[n]:.6f}" if n>0 else 
    f"& a_{{{n}}} = {an_F[n]:.6f}  & &" 
    for n in range(0, Nmax + 1)
]

display(Markdown(f""" 
$$
\\begin{{aligned}}
{"\\\\\n".join(title_line + lines)}
\\end{{aligned}}
$$
"""))




Fourier coefficients for the mystery function $F$:

 
$$
\begin{aligned}
& \cos\quad \text{coefficients}&&\sin \quad\text{coefficients}\\
& a_{0} = -0.000000  & &\\
&a_{1} = 0.000000  & &b_{1} = -0.317041\\
&a_{2} = -0.000000  & &b_{2} = 0.246588\\
&a_{3} = -0.000000  & &b_{3} = -0.176134\\
&a_{4} = 0.000000  & &b_{4} = 0.246588\\
&a_{5} = 0.000000  & &b_{5} = -0.317041
\end{aligned}
$$


In [4]:


# Plot: Fitting the model again, now with the coefficients recovered from integrals.
NmaxPrint = 5

fig = Figure(x_range=(0, 1/220), y_range=(-2, 2), samples=1200, autonormalization=True)
fig.show()

dynamic_model=a[0] + sum(
    a[n] * cos(2*pi*n*x) + b[n] * sin(2*pi*n*x) if n>0 else  a[n] * cos(2*pi*n*x) 
    for n in range(1, NmaxPrint + 1)
)

with fig:
    set_title("Recovered Fourier model for $F$")
    for n in range(0, NmaxPrint+1):
        parameter(a[n],min=-1,max=+1,value=0)
        if n>0:
            parameter(b[n],min=-1,max=+1,value=0)
    plot(F(220 * x), x, label=r"$F(220 x)$", id="F", width=4)
    plot(dynamic_model.subs(x, 220*x), x, label=r"$\mathrm{model}$", id="model", width=3, autonormalization=True)
    for j in range(0, NmaxPrint+1):
        plot(a[j] * cos(2*pi * j * 220 * x), x,
             label=rf"$a_{j}\sin(2\pi \cdot {j} \cdot 220 x)$",
             color="green", alpha=0.2)
        if j>0:
            plot(b[j] * sin(2*pi * j * 220 * x), x,
                 label=rf"$b_{j}\sin(2\pi \cdot {j} \cdot 220 x)$",
                 color="green", alpha=0.2)

TypeError: Figure() got unexpected keyword argument(s): autonormalization

### The *square wave* $\operatorname{Sq}$

In [ ]:
# Compute Fourier coefficients of the mystery function using explicit numerical integration.
Nmax = 100
an_Sq, bn_Sq = fourier_coefficients(Sq(x), Nmax, interval=(0, 1))

## Display coefficients
display(Markdown("Fourier coefficients for the square wave $Sq$:"))

title_line=[
    f"& \\cos\\quad \\text{{coefficients}}&&\\sin \\quad\\text{{coefficients}}"
]
NmaxPrint=10
lines = [
    f"&a_{{{n}}} = {an_Sq[n]:.6f}  & &b_{{{n}}} = {bn_Sq[n]:.6f}" if n>0 else 
    f"& a_{{{n}}} = {an_Sq[n]:.6f}  & &" 
    for n in range(0, NmaxPrint + 1)
]

display(Markdown(f""" 
$$
\\begin{{aligned}}
{"\\\\\n".join(title_line + lines)}
\\end{{aligned}}
$$
"""))


In [ ]:
# Plot: Fitting the model again, now with the coefficients recovered from integrals.
fig = Figure(x_range=(0, 1/220), y_range=(-2, 2), samples=1200,autonormalization=True)
fig.show()
NmaxPrint=8
dynamic_model=a[0] + sum(
    a[n] * cos(2*pi*n*x) + b[n] * sin(2*pi*n*x) if n>0 else  a[n] * cos(2*pi*n*x) 
    for n in range(1, NmaxPrint + 1)
)
with fig:
    set_title("Recovered Fourier model for $\\operatorname{Sq}$")
    for n in range(0, NmaxPrint+1):
        parameter(a[n],min=-1,max=+1,value=0)
        if n>0:
            parameter(b[n],min=-1,max=+1,value=0)
    plot(Sq(220 * x), x, label=r"$\operatorname{Sq}(220 x)$", id="Sq", width=4)
    plot(dynamic_model.subs(x, 220*x), x, label=r"$\mathrm{model}$", id="model", width=3, autonormalization=True)
    for j in range(0, NmaxPrint+1):
        plot(a[j] * cos(2*pi * j * 220 * x), x,
             label=rf"$a_{j}\sin(2\pi \cdot {j} \cdot 220 x)$",
             color="green", alpha=0.2)
        if j>0:
            plot(b[j] * sin(2*pi * j * 220 * x), x,
                 label=rf"$b_{j}\sin(2\pi \cdot {j} \cdot 220 x)$",
                 color="green", alpha=0.2)

## Now we make it automatic

In [ ]:
Nmax=100

an_Sq, bn_Sq = fourier_coefficients(Sq(x), Nmax)


lines = [
    f"&a_{{{n}}} = {an_Sq[n-1]:.6f}  & &b_{{{n}}} = {bn_Sq[n-1]:.6f}"
    for n in range(0, 10 + 1)
]

display(Markdown(f""" 
$$
\\begin{{aligned}}
{"\\\\\n".join(lines)}
\\end{{aligned}}
$$
"""))

## TODO: introduce MD as alias to display(Markdown()), but if provided a sequence be smart about it. If strings then display as concatenated markdown (with one newline separating the strings)
## If there are rich objects then try to display them e.g. sympy -> latex (inline), Dataframe -> block display table (use display())

## The partial sum, n from 0 to N
N=symbols("N")
SN_Sq=an_Sq[0]+ sum( [
        Piecewise((1,n<=N),(0,True)) * an_Sq[n]*sin(2*pi*n*x)
        +
        Piecewise((1,n<=N),(0,True)) * bn_Sq[n]*sin(2*pi*n*x)
    for n in range(1,Nmax+1)
    ])    

#TODO Introduce the functions One(condition) that is Piecewise((1,condition),(0,True))